# Privacy Evaluation of Tabular Data

In [4]:
from pathlib import Path
import os

# Make sure we are in the root directory
def set_project_root(marker="pyproject.toml"):
    path = Path.cwd()
    for parent in [path, *path.parents]:
        if (parent / marker).exists():
            os.chdir(parent)
            return parent
    raise FileNotFoundError(f"Could not find {marker} in any parent directory")
set_project_root()

from logging import INFO
from typing import Any

from hydra import initialize, compose
import pandas as pd
from omegaconf import DictConfig, OmegaConf
import json

from midst_toolkit.common.logger import log
from midst_toolkit.data_processing.midst_data_processing import load_midst_data_with_test
from midst_toolkit.evaluation.metrics_base import MetricBase

from midst_toolkit.evaluation.privacy import (
    DistanceToClosestRecordScore,
    EpsilonIdentifiabilityRisk,
    HittingRate,
    MedianDistanceToClosestRecordScore,
    NearestNeighborDistanceRatio,
)

from midst_toolkit.evaluation.privacy.distance_preprocess import preprocess_for_distance_computation
from midst_toolkit.evaluation.privacy.distance_utils import NormType
from midst_toolkit.evaluation.privacy.epsilon_identifiability_risk import EpsilonIdentifiabilityNorm
from midst_toolkit.evaluation.quality.confidence_interval_overlap import ConfidenceLevel

# Local imports
from implementations.tabular_data.evaluation.preprocessing import (   
    get_numerical_and_categorical_column_names,
    preprocess_data_for_alpha_precision_eval,
    syntheval_preprocess,
)
from implementations.tabular_data.evaluation.display_utils import log_metrics

## Load the data for quality, utility and privacy evaluation
Evaluation also needs a JSON file containing meta information about the dataset. The provided `meta_info.json` JSON file should provide information about the columns as well as the target column, specifically which columns correspond to numerical and categorical values, which column corresponds to a label (if any), and the downstream task to be performed with this dataset.
The provided pre-processing script (`preprocess_berka_trans.py`), generates this JSON file for the transaction table in the Berka dataset, and the evaluation pipeline loads it as a dictionary. Here is an example of `meta_info`.
```bash
}
    "num_col_idx": [0,3,4,7],
    "cat_col_idx": [1,2,5,6],
    "target_col_idx": [1],
    "task_type": "multiclass"
}
```
Types of supported tasks: "binclass","multiclass", and "regression"

Other dataframes to load:

- `real_train_data`: real data that is used to train the generative model

- `real_holdout_data`: real data that is NOT used in training

- `synthetic_data`: the synthesized data 



In [8]:
ROOT = Path.cwd()
IMPLEMENTATION_ROOT = ROOT / "implementations" / "tabular_data" / "single_table" 
# Set data and output directories
base_data_dir = IMPLEMENTATION_ROOT / "data"
base_output_dir = IMPLEMENTATION_ROOT / "results"

TABLE_NAME = "trans"

real_train_data = pd.read_csv(base_data_dir / f"{TABLE_NAME}.csv")
real_holdout_data = pd.read_csv(base_data_dir / f"{TABLE_NAME}_holdout.csv")
synthetic_data = pd.read_csv(base_output_dir / "single_table_synthesizing/trans/_final"/ f"{TABLE_NAME}_synthetic.csv")

with open(base_data_dir / "meta_info.json", "r") as f:
    meta_info = json.load(f)

log(INFO, f"Loaded {TABLE_NAME} data for evaluations")
log(INFO, f"Loaded meta_info for {TABLE_NAME} data")
log(INFO, f"Loaded {len(real_train_data)} rows of real training data")
log(INFO, f"Loaded {len(real_holdout_data)} rows of real holdout data")
log(INFO, f"Loaded {len(synthetic_data)} rows of synthetic data")

INFO :      Loaded trans data for evaluations
INFO :      Loaded meta_info for trans data
INFO :      Loaded 16000 rows of real training data
INFO :      Loaded 4000 rows of real holdout data
INFO :      Loaded 3200 rows of synthetic data


## Pre-process the data for evaluation
Preprocessing can have a noticeable impact on the metrics. As such, it is important to handle such transformations with care and to take into account the calculations being performed in the metrics themselves. For example, if computing distance-based metrics with Euclidean norms, categorical variables should be one-hot encoded, rather than ordinally encoded. This notebook handles proper preprocessing of the datasets before applying the various metrics in the pipeline.

In this notebook:

- When applying the distance based metrics of `DistanceToClosestRecordScore`, `MedianDistanceToClosestRecordScore`, and `NearestNeighborDistanceRatio`, categorical variables are one-hot encoded and numerical variables are normalized by their range such that values are guaranteed to fall between [-1, 1].

- For all other metrics, which leverage the `SynthEval` library to perform at least some portion of the computations, the standard SynthEval preprocessing pipeline is applied. This process ordinally encodes categorical variables and min-max encodes numerical variables. `SynthEval` library: https://github.com/schneiderkamplab/syntheval

In [9]:
# Shared preprocessing for syntheval based metrics if they are to be run
log(INFO, "Preprocessing Data with SynthEval pipeline")
numerical_columns, categorical_columns = get_numerical_and_categorical_column_names(real_train_data, meta_info)
# Categorical values are ordinal encoded, numerical values are min-max scaled
syntheval_real_data_train, syntheval_synthetic_data, syntheval_real_data_holdout = syntheval_preprocess(
    numerical_columns, categorical_columns, real_train_data, synthetic_data, real_holdout_data
)

INFO :      Preprocessing Data with SynthEval pipeline


## Hitting Rate
This metric, determines the Hitting (Exact Match) rate associated with real and synthetic data. The rate is computed as the percentage of real data points provided that are "replicated" within the provided synthetic data. **A smaller rate is better**.

**hitting_threshold parameter:**
A synthetic data point is considered to "replicate" a real data point if each of its numerical values
are within ``hitting_threshold`` percent of the variable range in the real data. For each categorical value an
exact match is required.


**Pre-processing note:** 
Categorical variables must be encoded in some way (ordinal or vector) for the evaluation to work. This can be accomplished by preprocessing the dataframes before calling compute or by setting ``do_preprocess`` to True.

In [11]:
log(INFO, "Running Hitting Rate Evaluation")
# Hyper-parameter
hitting_threshold = 0.03

metric = HittingRate(
    categorical_columns=categorical_columns,
    numerical_columns=numerical_columns,
    hitting_threshold=hitting_threshold,
    # Already preprocessing above
    do_preprocess=False,
)
results = metric.compute(syntheval_real_data_train, syntheval_synthetic_data)
results

INFO :      Running Hitting Rate Evaluation


{'hitting_rate': 0.000125}

## Epsilon Identifiability Rate

Epsilon Identifiability Risk computes the ratio of real data points that have a synthetic data point closer than any other real data point in the set of data points. As such, **a value closer to 0 is better**.

If a holdout set is provided to the compute function, the same ratio is computed for holdout data points compared with synthetic ones. The difference between the ratio for the real data points compared with the holdout data points is then calculated. Ideally, these should be roughly the same (i.e. difference near zero) or negative. 


NOTE: Columns are not uniformly weighted. They are weighted by their inverse column entropy to provide greater attention to rare data points. This is formally defined in:

Yoon, J., Drumright, L.N., Schaar, M.: Anonymization through data synthesis using generative adversarial
networks (ADS-GAN). IEEE J. Biomed. Health Informatics 24(8), 2378–2388 (2020)
https://doi.org/10.1109/JBHI.2020.2980262


**Pre-process**:
The dataframes provided need to be pre-processed into numerical values for each column in some way. That
is, for example, the categorical variables may be one-hot encoded and the numerical values normalized in
some way.


In [12]:
log(INFO, "Running Epsilon Identifiability Rate Evaluation")
# Categorical values are ordinal encoded, numerical values are min-max scaled
metric = EpsilonIdentifiabilityRisk(
    categorical_columns=categorical_columns,
    numerical_columns=numerical_columns,
    norm=EpsilonIdentifiabilityNorm("gower"), # Options are "euclid" and "gower".
    # Already preprocessing above
    do_preprocess=False,
)
results = metric.compute(syntheval_real_data_train, syntheval_synthetic_data)
results

INFO :      Running Epsilon Identifiability Rate Evaluation
Computing nearest neighbor distances from real/holdout dataset to synthetic dataset.:   0%|          | 0/25 [00:00<?, ?it/s]/Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/.venv/lib/python3.12/site-packages/syntheval/utils/nn_distance.py:137: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  b[num_cols] = b[num_cols].astype("float")
Computing nearest neighbor distances from real/holdout dataset to synthetic dataset.:   4%|▍         | 1/25 [00:00<00:02,  9.62it/s]/Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/.venv/lib/python3.12/site-packages/syntheval/utils/nn_distance.py:137: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slic

{'epsilon_identifiability_risk': np.float64(0.000125)}

## Distance-Based metrics
### Pre-processing
For the distance-based metrics, categorical variables are one-hot encoded and numerical variables are normalized by their range such that values are guaranteed to fall between [-1, 1].

In [13]:
log(INFO, "Preprocessing Data for Distance Evaluation")
# Categorical values are one-hot encoded, numerical values are scaled by their range, but not into [0,1]
distance_real_data, distance_synthetic_data, distance_holdout_data = preprocess_for_distance_computation(
    meta_info=meta_info,
    real_data_train=real_train_data,
    synthetic_data=synthetic_data,
    real_data_test=real_holdout_data,
)

INFO :      Preprocessing Data for Distance Evaluation


### Distance To Closest RecordScore (DCR) Metric

DCR is defined as the distance between a synthetic datapoint and its nearest real datapoint. DCR equal to zero means that the synthetic data is at a higher risk of privacy leakage, while **higher DCR values mean less risk of privacy leakage**.

This class computes the DCR of each synthetic datapoint to real data points in two different sets:

- Training (real) data used to train the model that generated the synthetic data.
- Holdout (real) data from the same distribution as the training data but that was NOT used to train the model.

It returns the proportion of synthetic data points that are closer to the training dataset than the
holdout dataset. If the size of the training and holdout datasets are equal, this score should ideally be
indicating that the model has not over fit to training data and the synthetic data points are not memorized
copies of training data. If the size of the training and holdout datasets are different, the ideal value for
this score is # ``real_data`` / (# ``real_data`` + # ``holdout_data``).

In [14]:
log(INFO, "Running DCR Evaluation")
# Hyper-parameters
norm = "l2" # Options are "l1" and "l2"
batch_size = 1000
metric = DistanceToClosestRecordScore(NormType(norm), batch_size, do_preprocess=False)
results = metric.compute(distance_real_data, distance_synthetic_data, distance_holdout_data)
results

INFO :      Running DCR Evaluation
100%|██████████| 4/4 [00:02<00:00,  1.79it/s]
INFO :      Distance to Closest Record Score = 0.8128125071525574


{'dcr_score': 0.8128125071525574}